##### Stage 1

In [ ]:
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step, Event

class MyWorkflow(Workflow):
    @step
    async def my_step(self, ev: StartEvent ) -> StopEvent:
        return StopEvent(result=f'Welcome to LlamaIndex {ev.name}')
    
w = MyWorkflow(timeout=10, verbose=True)
result = await w.run(name='Farah')
print(result)

[tick] add: StartEvent(name='Farah')
[my_step:0] started from StartEvent
[result] StopEvent(result='Welcome to LlamaIndex Farah')
[my_step:0] complete with StopEvent
Welcome to LlamaIndex Farah


##### Stage 2

In [3]:
from llama_index.core.workflow import StartEvent, StopEvent, step, Event, Workflow

class MyCustomEvent(Event):
    text_process: str

class Myworkflow_two(Workflow):
    @step
    async def step_one(self, event: StartEvent) -> MyCustomEvent:
        text = event.text
        capitalize = text.title()
        return MyCustomEvent(text_process=capitalize)
    
    @step
    async def step_two(self, event: MyCustomEvent) -> StopEvent:
        text = event.text_process
        return StopEvent(result=f'{text} 😊')
    

w_two = Myworkflow_two(timeout=20, verbose=True)
result = await w_two.run(text='I love coding')
print(result)

[tick] add: StartEvent(text='I love coding')
[step_one:0] started from StartEvent
[step_one:0] complete with MyCustomEvent
[tick] add: MyCustomEvent(text_process='I Love Coding')
[step_two:0] started from MyCustomEvent
[result] StopEvent(result='I Love Coding 😊')
[step_two:0] complete with StopEvent
I Love Coding 😊


##### Stage 3

In [7]:
from llama_index.core.workflow import StartEvent, StopEvent, Event, step, Workflow

class LoopEvent(Event):
    number: int

class Myworkflow_three(Workflow):
    @step
    async def step_one(self, ev: StartEvent | LoopEvent) -> StopEvent | LoopEvent:
        if ev.number % 2 != 0:
            print("Odd Number Adding 1")
            ev.number +=1
            return LoopEvent(number=ev.number)
        
        return StopEvent(result=f'Final Even Number is {ev.number}')
    

w = Myworkflow_three(timeout=20, verbose=True)
result = await w.run(number=3)
print(result)

[tick] add: StartEvent(number=3)
[step_one:0] started from StartEvent
Odd Number Adding 1
[step_one:0] complete with LoopEvent
[tick] add: LoopEvent(number=4)
[step_one:0] started from LoopEvent
[result] StopEvent(result='Final Even Number is 4')
[step_one:0] complete with StopEvent
Final Even Number is 4


##### Stage 4

In [ ]:
from llama_index.core.workflow import StartEvent, StopEvent, Event, Workflow, step, Context

class add_n1_n2(Event):
    num: int
    num_3: int

class add_n3(Event):
    num_ : int

class Myworkflow_four(Workflow):
    @step
    async def add_num1_num2(self, ctx: Context ,ev: StartEvent) -> add_n1_n2:
        await ctx.store.set('username', ev.username)

        return add_n1_n2(num=ev.n1 + ev.n2, num_3=ev.n3)

    @step 
    async def add_num3(self, ctx: Context, ev: add_n1_n2) -> add_n3:
        return add_n3(num_=ev.num + ev.num_3)

    @step
    async def final(self, ctx: Context, ev: add_n3) -> StopEvent:
        username = await ctx.store.get('username')
        return StopEvent(result=f'{username} your final sum is: {ev.num_}')
    
w = Myworkflow_four(timeout=20, verbose=True)
result = await w.run(username='Raha', n1=45, n2=2, n3=72)
print(result)

[tick] add: StartEvent(username='Raha', n1=45, n2=2, n3=72)
[add_num1_num2:0] started from StartEvent
[add_num1_num2:0] complete with add_n1_n2
[tick] add: add_n1_n2(num=47, num_3=72)
[add_num3:0] started from add_n1_n2
[add_num3:0] complete with add_n3
[tick] add: add_n3(num_=119)
[final:0] started from add_n3
[result] StopEvent(result='Raha your final sum is: 119')
[final:0] complete with StopEvent
Raha your final sum is: 119
